# 파티 코치 LoRA 파인튜닝 — 노트북 · RTX 4050 6GB

Qwen3.5-4B + Unsloth. **4bit QLoRA (bf16 계산), 최대 길이 4096토큰**.
같은 내용의 다른 버전: `finetune_laptop`(RTX 4050) · `finetune_home`(RTX 3060) · `finetune_colab`(T4). 설정 셀만 다르다.

`llm/agent`의 **시스템 프롬프트·도구 정의를 그대로** 써서, 모델이 도구 호출 방식과 파티 판단을 배우게 한다.
사실(종족값·사용률 등)은 외우게 하지 않는다. 학습 데이터 속 도구 결과는 항상 실제 back API 값이다.

> 6GB라 4096토큰까지만 넣는다. 샘플 만들기 대화(약 3,100~4,500토큰)는 들어가지만 파티 빌딩 대화(약 5,500토큰)는 6번 셀에서 빠진다. 파티 빌딩까지 학습하려면 집 PC나 Colab 버전을 쓴다.

**준비**
- 학습 데이터: Windows에서 `cd llm` → `..\.venv\Scripts\python training\make_data.py --n 1000`
  → `llm/training/train.jsonl`, `eval.jsonl` (DB의 실제 파티·샘플로 생성, git 제외)
- WSL2(Ubuntu)에서 venv를 따로 만들고 `pip install unsloth jupyter httpx` (Windows의 `.venv`와 섞지 않는다)
- 저장소의 `llm/training/`에서 노트북을 연다. `llm/agent/prompt.py`, `tools.py`는 자동으로 찾는다.

In [ ]:
import os, sys, json, re, subprocess
from pathlib import Path

subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'])

## 1. 설치
WSL2 venv에 `pip install unsloth jupyter httpx`를 한 번 해 두면 된다 (이 노트북에서는 설치하지 않음).

## 2. 설정
이 버전의 GPU에 맞춘 값이다. 다른 GPU에서 열었으면 경고가 뜬다.

In [ ]:
import torch

MODEL_NAME = 'unsloth/Qwen3.5-4B'          # 원본: Qwen/Qwen3.5-4B (Apache 2.0, 이미지도 받는 모델이라 FastModel로 불러옴)
LOAD_IN_4BIT = True
MAX_SEQ_LEN = 4096                     # 넘는 대화는 학습에서 뺀다 (6번 셀). 시스템 프롬프트+도구 설명만 약 2,400토큰
BF16 = torch.cuda.is_bf16_supported()     # T4는 False → fp16

LORA_R = 16
EPOCHS = 2
LR = 2e-4
BATCH, GRAD_ACC = 1, 8                     # 실제 배치 = BATCH × GRAD_ACC

DATA_DIR = Path('.')
OUT_DIR = Path('outputs')                  # llm/training/outputs (git 제외)

gpu = torch.cuda.get_device_properties(0)
VRAM_GB = gpu.total_memory / 2**30
if 'RTX 4050 Laptop' not in gpu.name:
    print(f'⚠ 이 노트북은 RTX 4050 Laptop용인데 지금 GPU는 {gpu.name} ({VRAM_GB:.1f}GB). 맞는 버전을 쓰거나 위 값을 고칠 것')
print(f'{gpu.name} {VRAM_GB:.1f}GB | 4bit={LOAD_IN_4BIT} bf16={BF16} seq={MAX_SEQ_LEN} out={OUT_DIR}')

## 3. 시스템 프롬프트·도구 불러오기
서버(`llm/agent`)와 같은 프롬프트·도구로 학습해야 붙였을 때 그대로 동작한다.
도구 정의는 Claude 형식(`input_schema`)이라 Qwen 형식(`function.parameters`)으로 바꾼다.

In [ ]:
import importlib.util

def find_agent_dir():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / 'llm/agent/tools.py').exists():
            return base / 'llm/agent'
    raise FileNotFoundError('저장소 안(llm/training/)에서 노트북을 열 것')

def load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

AGENT = find_agent_dir()
prompt = load('prompt', AGENT / 'prompt.py')
tools_mod = load('tools', AGENT / 'tools.py')        # httpx 필요 (Colab엔 기본 설치)

SYSTEM = prompt.SYSTEM
TOOLS = [{'type': 'function', 'function': {'name': t['name'], 'description': t['description'],
                                           'parameters': t['input_schema']}} for t in tools_mod.TOOLS]
print(AGENT, '| 도구', len(TOOLS), '개:', ', '.join(t['function']['name'] for t in TOOLS))

## 4. 학습 데이터
`make_data.py`가 만든 `train.jsonl`(학습)과 `eval.jsonl`(8번 셀 평가용). 한 줄 = 대화 하나.
```json
{"kind": "fill", "format": "doubles", "messages": [
  {"role": "user", "content": "[화면] 포맷: 더블 (doubles), ...\n\n나머지 1마리 채워줘"},
  {"role": "assistant", "content": "", "tool_calls": [{"type": "function", "function": {"name": "analyze_party", "arguments": {...}}}]},
  {"role": "tool", "content": "<실제 도구 결과 JSON>"},
  ...
  {"role": "assistant", "content": "최종 답변"}
]}
```
시스템 프롬프트는 파일에 없고 여기서 넣는다.

In [ ]:
from collections import Counter

def read(name):
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f'{path} 없음 — make_data.py로 만들어서 ' + ('올릴 것' if str(DATA_DIR) == '/content' else '이 폴더에 둘 것'))
    rows = [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]
    for r in rows:
        if r['messages'][0]['role'] != 'system':
            r['messages'].insert(0, {'role': 'system', 'content': SYSTEM})
    return rows

train_rows, eval_rows = read('train.jsonl'), read('eval.jsonl')
print(f'학습 {len(train_rows)}개 / 평가 {len(eval_rows)}개')
print(Counter(r['kind'] for r in train_rows))

## 5. 모델 불러오기 + LoRA
Qwen3.5는 이미지도 받는 모델이라 `FastModel`로 불러오고, 언어 부분에만 LoRA를 붙인다.

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN, load_in_4bit=LOAD_IN_4BIT, full_finetuning=False)
tok = getattr(tokenizer, 'tokenizer', tokenizer)     # processor면 안쪽 글자 tokenizer

model = FastModel.get_peft_model(
    model, finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=3407)
model.print_trainable_parameters()

## 6. 채팅 템플릿 적용
Qwen3.5 템플릿이 도구 정의는 시스템 프롬프트에, 도구 호출은 `<tool_call><function=이름><parameter=키>값</parameter>...`,
결과는 user 차례의 `<tool_response>`로 넣는다. thinking은 끈다 (서버에서도 끄고 쓸 것).

In [ ]:
from datasets import Dataset

def render(messages, add_generation_prompt=False):
    return tok.apply_chat_template(messages, tools=TOOLS, tokenize=False,
                                   add_generation_prompt=add_generation_prompt, enable_thinking=False)

texts = [render(r['messages']) for r in train_rows]
lens = [len(tok(t).input_ids) for t in texts]
keep = [t for t, n in zip(texts, lens) if n <= MAX_SEQ_LEN]
dropped = Counter(r['kind'] for r, n in zip(train_rows, lens) if n > MAX_SEQ_LEN)
print(f'토큰 길이 최대 {max(lens)}, 평균 {sum(lens) // len(lens)} / {MAX_SEQ_LEN} 넘어서 뺀 것 {len(texts) - len(keep)}개 {dict(dropped)}')
dataset = Dataset.from_dict({'text': keep})
print(keep[0][-1200:])

## 7. 학습
답변(assistant) 부분만 loss를 계산한다. 사용자 질문·도구 결과(`<tool_response>`, user 차례로 들어감)는 제외.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field='text', output_dir=str(OUT_DIR / 'checkpoints'),
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC,
        num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type='cosine', warmup_ratio=0.05,
        optim='adamw_8bit', weight_decay=0.01, bf16=BF16, fp16=not BF16,
        logging_steps=5, save_strategy='epoch', save_total_limit=1, seed=3407, report_to='none'),
)
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n',
                                  response_part='<|im_start|>assistant\n')
stats = trainer.train()
print(f"{stats.metrics['train_runtime'] / 60:.1f}분, 최대 VRAM {torch.cuda.max_memory_reserved() / 2**30:.1f}GB")

## 8. 평가: 첫 도구를 맞게 부르나
`eval.jsonl`(학습에 안 쓴 대화)에서 사용자 질문만 주고, 모델이 처음 부른 도구가 정답 대화의 첫 도구와 같은지 센다.
실제 도구 실행까지는 서버(`llm/agent/loop.py`)에 붙여서 확인.

In [ ]:
TOOL_CALL = re.compile(r'<tool_call>\s*<function=([\w-]+)>(.*?)</function>\s*</tool_call>', re.S)
PARAM = re.compile(r'<parameter=([\w-]+)>\n?(.*?)\n?</parameter>', re.S)

def parse_calls(text):
    """Qwen3.5 도구 호출 → [(이름, 인자)]. 값이 JSON이면 풀어서."""
    calls = []
    for name, body in TOOL_CALL.findall(text):
        args = {}
        for k, v in PARAM.findall(body):
            try:
                args[k] = json.loads(v)
            except json.JSONDecodeError:
                args[k] = v
        calls.append((name, args))
    return calls

def first_turn(msgs, max_new_tokens=400):
    ids = tok(render(msgs, add_generation_prompt=True), return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, temperature=0.7, top_p=0.8, do_sample=True)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=False)

model.eval()
hit = Counter(); total = Counter()
for r in eval_rows[:40]:
    gold = r['messages'][2]['tool_calls'][0]['function']['name']
    text = first_turn(r['messages'][:2])
    calls = parse_calls(text)
    total[r['kind']] += 1
    hit[r['kind']] += bool(calls) and calls[0][0] == gold
for k in total:
    print(f'{k}: 첫 도구 정답 {hit[k]}/{total[k]}')
print(f'전체 {sum(hit.values())}/{sum(total.values())}')
print('\n마지막 예시 출력:\n', text[:600])

## 9. 저장
LoRA 어댑터만 저장한다 (수십 MB). 서버에서는 원본 모델 + 어댑터로 불러 쓴다.

In [ ]:
ADAPTER = OUT_DIR / 'qwen35-4b-coach-lora'
model.save_pretrained(ADAPTER)
tokenizer.save_pretrained(ADAPTER)
print('저장:', ADAPTER)

### (선택) 병합·GGUF
llama.cpp/Ollama로 돌리려면 4bit GGUF로 내보낸다 (추론 약 3GB). 시간이 오래 걸리니 필요할 때만.
```python
model.save_pretrained_gguf(str(OUT_DIR / 'gguf'), tokenizer, quantization_method='q4_k_m')
```